# DINOv2 Frozen Probe — Contrail Segmentation (Google Colab)

Baseline: fully frozen DINOv2 backbone, MLP head trained on top of intermediate features.  
No VPT — backbone weights are never updated.

**Prerequisites:**
- Runtime → Change runtime type → **T4 GPU**
- Secrets panel (🔑): add `WANDB_API_KEY` and `HF_TOKEN`
- Google Drive with data at `MyDrive/cv data/`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo & install dependencies

In [ ]:
import os, subprocess, sys, torch

REPO = '/content/contrail-segmentation'

# record Colab's pre-installed torch before deps can overwrite it
_torch_ver  = torch.__version__.split('+')[0]          # e.g. "2.6.0"
_cuda_tag   = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else None

if not os.path.exists(REPO):
    os.system(f'git clone https://github.com/aryangarg794/contrail-segmentation.git {REPO}')
os.system(f'cd {REPO} && git checkout attention-unet-training && git pull')

# install deps — lightning/transformers may pull in a cpu torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scipy>=1.15.0',
                'lightning', 'transformers>=5.3.0', 'timm',
                'albumentations',
                'segmentation-models-pytorch',
                'hydra-core', 'wandb', 'dill', 'rich', 'tqdm'], check=True)

# restore Colab's exact CUDA torch (correct SM kernels for this GPU)
if _cuda_tag:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f'torch=={_torch_ver}', 'torchvision',
                    '--index-url', f'https://download.pytorch.org/whl/{_cuda_tag}'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
               cwd=REPO, check=True)

import importlib
importlib.invalidate_caches()
sys.path.insert(0, f'{REPO}/src')

import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
x = torch.tensor([1.0]).cuda()
print(f'CUDA test: PASSED {x}')

## 3. Mount Google Drive & set data paths

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

TRAIN_DATA = '/content/drive/MyDrive/cv data/train'
METADATA   = '/content/drive/MyDrive/cv data/metadata.csv'
print('Train dir exists:', os.path.exists(TRAIN_DATA))
print('Metadata exists: ', os.path.exists(METADATA))

## 4. Patch data paths at runtime

In [ ]:
import pandas as pd
import contrail_segmentation.data.utils as data_utils

data_utils.DATA_DIR  = TRAIN_DATA
data_utils.META_PATH = METADATA

metadata  = pd.read_csv(METADATA)
available = set(os.listdir(TRAIN_DATA))
metadata  = metadata[metadata['record_id'].apply(lambda x: str(int(x))).isin(available)].reset_index(drop=True)
data_utils.metadata = metadata

print(f'Dataset size: {len(data_utils.metadata)} records ({len(available)} on disk)')

## 5. W&B login

Add `WANDB_API_KEY` in the Colab secrets panel (🔑 icon in the left sidebar).

In [ ]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))

## 6. Build preprocessing transforms

Add `HF_TOKEN` in the Colab secrets panel (🔑 icon in the left sidebar).

In [ ]:
import albumentations as A
from transformers import AutoImageProcessor
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

MODEL_NAME = 'facebook/dinov2-large'

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
mean, std = processor.image_mean, processor.image_std
print(f'DINOv2 norm  mean={mean}  std={std}')

train_transform = A.Compose([
    A.ShiftScaleRotate(scale_limit=0.2, rotate_limit=180, shift_limit=0.3,
                       border_mode=0, value=0, p=0.5),
    A.HorizontalFlip(),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.5),
    A.Normalize(mean=mean, std=std),
])

val_transform = A.Compose([
    A.Normalize(mean=mean, std=std),
])

## 7. Build dataloaders

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from contrail_segmentation.data.dataset import ContrailDataset

SEED        = 0
BATCH_SIZE  = 16
NUM_WORKERS = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
generator = torch.Generator().manual_seed(SEED)

full_dataset = ContrailDataset(mask_only=True)
indices      = np.arange(len(full_dataset))
np.random.shuffle(indices)
train_size   = int(0.8 * len(indices))
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_set = ContrailDataset(mask_only=True, transform=train_transform)
val_set   = ContrailDataset(mask_only=True, transform=val_transform)

train_loader = DataLoader(Subset(train_set, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          pin_memory=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(Subset(val_set, val_idx), batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=NUM_WORKERS)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

## 8. Build model

In [ ]:
from contrail_segmentation.models.dino_probe import DINOv2Probe

model = DINOv2Probe(
    model_name=MODEL_NAME,
    lr=1e-4,
    wd=1e-3,
    threshold=0.5,
    tversky_alpha=0.3,
    tversky_beta=0.7,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)')

## 9. Train

In [ ]:
from datetime import datetime
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import WandbLogger

MAX_EPOCHS = 100
timestamp  = datetime.now().strftime('%d_%b_%Y__%Hh%Mm')
run_name   = f'dinov2_probe_seed{SEED}_{timestamp}'

logger = WandbLogger(project='contrail-segmentation', name=run_name, save_dir='/content/wandb_logs')

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    precision='16-mixed',
    log_every_n_steps=1,
    logger=logger,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

## 10. Find best threshold & test

In [ ]:
from contrail_segmentation.train.utils import find_best_threshold

best_thresh = find_best_threshold(model, val_loader)
model.threshold = best_thresh
model.mask_only = True

test_metrics = trainer.test(model, dataloaders=val_loader)
print(test_metrics)

## 11. Save checkpoint to Drive

In [ ]:
save_path = f'/content/drive/MyDrive/cv data/checkpoints/dinov2_probe_{timestamp}.pt'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f'Saved to {save_path}')